In [17]:
from llm import LLM
from tokenizer import Tokenizer
import torch.nn as nn
import torch
import json
from data_types import Config

In [18]:
device = 'cpu'
num_episodes = 10

In [19]:
seinfeld_episodes = json.load(open('seinfeld_scripts.json', 'r'))

In [20]:
episode_list = []
for season in seinfeld_episodes.keys():
    for episode in seinfeld_episodes[season].keys():
        episode_list.append((" ".join(seinfeld_episodes[season][episode].split()))[1:])

In [21]:
training_episodes = episode_list[:num_episodes]
all_episodes =" ".join(episode_list)

In [22]:
tokenizer = Tokenizer()
all_tokens = tokenizer.process_set_tokenize_text(all_episodes)
print(all_tokens[:100])

['good', 'news', ',', 'bad', 'news', 'written', 'by', 'larry', 'david', '', '', 'jerry', 'seinfeld', '(', 'comedy', 'club', ')', '', 'jerry', 'you', 'know', ',', 'why', 'we', 're', 'here', '?', 'to', 'be', 'out', ',', 'this', 'is', 'out', '.', '.', '.', 'and', 'out', 'is', 'one', 'of', 'the', 'single', 'most', 'enjoyable', 'experiences', 'of', 'life', '.', 'people', '.', '.', '.', 'did', 'you', 'ever', 'hear', 'people', 'talking', 'about', '"', 'we', 'should', 'go', 'out', '"', '?', 'this', 'is', 'what', 'they', 're', 'talking', 'about', '.', '.', '.', 'this', 'whole', 'thing', ',', 'we', 're', 'all', 'out', 'now', ',', 'no', 'one', 'is', 'home', '.', 'not', 'one', 'person', 'here', 'is', 'home', ',']


In [23]:
print(tokenizer.vocab_dict)

{'.': 0, '': 1, ',': 2, 'i': 3, 'the': 4, 'you': 5, '?': 6, 'jerry': 7, 's': 8, 'a': 9, 'to': 10, ')': 11, 'george': 12, '(': 13, 'it': 14, 'elaine': 15, '!': 16, 'kramer': 17, 'and': 18, 'that': 19, 't': 20, 'what': 21, 'of': 22, 'in': 23, '-': 24, 'is': 25, 'he': 26, 'on': 27, 'this': 28, 'me': 29, 'no': 30, 'm': 31, 'know': 32, 'oh': 33, 'with': 34, 'my': 35, 're': 36, 'yeah': 37, 'we': 38, 'for': 39, 'well': 40, 'don': 41, 'have': 42, 'so': 43, 'at': 44, 'do': 45, 'are': 46, 'up': 47, 'out': 48, 'they': 49, 'can': 50, 'was': 51, 'she': 52, 'his': 53, 'just': 54, 'not': 55, 'like': 56, 'all': 57, '"': 58, 'get': 59, 'there': 60, 'your': 61, 'her': 62, 'be': 63, 'hey': 64, 'about': 65, 'right': 66, 'him': 67, 'here': 68, 'got': 69, 'll': 70, 'go': 71, 'but': 72, 'how': 73, 'think': 74, 'see': 75, 'back': 76, 'one': 77, 'if': 78, 'uh': 79, 'from': 80, 'going': 81, 'why': 82, 'now': 83, 'did': 84, 'really': 85, 'look': 86, 'want': 87, 'good': 88, 'as': 89, 'gonna': 90, 'an': 91, 'who':

In [24]:
config = Config(200, len(tokenizer.vocab_arr), 50,device,10)
model = LLM(config,tokenizer)

In [25]:
print(len(tokenizer.vocab_arr))

19347


In [26]:
window_size = 20
batch_size = 50
data = torch.zeros((1, window_size+1), dtype=torch.long).to(device)
for episode in training_episodes:
    tokenized_episode = torch.tensor(tokenizer.process_tokenize_encode(episode))
    data = torch.cat([data, tokenized_episode.unfold(0, window_size+1, 1).to(device)], dim=0)

data = data[1:]
data_loader = torch.utils.data.DataLoader(data, batch_size=batch_size, shuffle=True)    


In [ ]:
num_epochs = 1
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
window_size = 50
for epoch in range(num_epochs):
        total_loss = 0
        total_n = 0
        for i, batch in enumerate(data_loader):
            optimizer.zero_grad()
            input_data = batch[:, :-1].to(device)
            output_data = batch[:, 1:].to(device)
            predicted_output = model(input_data)
            # CrossEntropyLoss expects: input (N, C), target (N,) with class indices
            loss = criterion(
                predicted_output.reshape(-1, config.d_vocab),  # (batch*seq, vocab_size)
                output_data.reshape(-1)                        # (batch*seq,) class indices
            )
            loss.backward()
            optimizer.step()
            total_loss += loss.item()*batch.shape[0]
            total_n += batch.shape[0]
            print(f'Progress: {i*batch_size}/{data.shape[0]}, loss= {loss.item()}', end='\r')
        print(f'Epoch: {epoch+1}/{num_epochs}, Average Loss: {(total_loss/total_n)}')


Epoch: 1/1, Average Loss: 2.8593448574692358447


In [28]:
model.generate("Jerry and George are sitting in the coffee shop, talking about their day. Jerry says, ", max_length=100)

'jerry and george are sitting in the coffee shop , talking about their day . jerry says , " ( jerry s sitting on his couch talking to a table . ) oh , he s a doctor . ( scene cuts to jerry )  george i m just telling you . you . you . you re really gonna do it .  george no , no , no , no , no , no , no , no , no , no . she won t take it . . . . . . . . . . . . . . .  jerry well , what does she do ?  george if'

In [29]:
print(torch.nn.functional.one_hot(torch.tensor([1,2,3]), config.d_vocab))


tensor([[0, 1, 0,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])


In [30]:
total_params = sum(p.numel() for p in model.parameters())


In [31]:
total_params

8774800